In [1]:
from openff.toolkit import Molecule, ForceField
from openff.toolkit.utils import get_data_file_path
import shutil, subprocess
from tqdm import tqdm

LICENSE: Could not open license file specified by OE_LICENSE environment variable "/Users/jeffreywagner/oe_license.txt"
LICENSE: Could not open license file "oe_license.txt" in local directory
LICENSE: N.B. OE_DIR environment variable is not set
LICENSE: No product keys!
LICENSE: No product keys!
LICENSE: No product keys!
The OpenEye Toolkits are found to be installed but not licensed and therefore will not be used.
The OpenEye Toolkits require a (free for academics) license, see https://docs.eyesopen.com/toolkits/python/quickstart-python/license.html
LICENSE: No product keys!
The OpenEye Toolkits are found to be installed but not licensed and therefore will not be used.
The OpenEye Toolkits require a (free for academics) license, see https://docs.eyesopen.com/toolkits/python/quickstart-python/license.html


In [2]:
from rdkit.Chem import AllChem
def gen_bad_conf(offmol):
    rdmol = offmol.to_rdkit()
    AllChem.EmbedMultipleConfs(
                rdmol,
                numConfs=1,
                #pruneRmsThresh=rms_cutoff.m_as(unit.angstrom),
                #randomSeed=1,
                useRandomCoords=True,
                useBasicKnowledge=False,
        useExpTorsionAnglePrefs=False,
        useMacrocycleTorsions=False
            )
    return Molecule.from_rdkit(rdmol)


In [3]:
import random
# dataset.smi from xtalpi fragments subset https://github.com/openforcefield/qca-dataset-submission/blob/master/submissions/2024-04-02-xtalpi-20-percent-fragments-torsiondrive-v1.0/dataset.smi
smileses = open('/Users/jeffreywagner/Downloads/dataset.smi').read().split()
random.shuffle(smileses)

In [4]:
! rm gen_conf* inputs.sdf *_min_* minimized_inputs.sdf

In [5]:
successes = 0

for smiles in smileses:
    if len(smiles) < 40:
        continue
    try:
        print(smiles)
        mol = Molecule.from_smiles(smiles)
        #mol.generate_conformers(n_conformers=1)
        mol = gen_bad_conf(mol)
        mol.to_file(f"gen_conf_{successes}.sdf", file_format='sdf')
        successes += 1
    except:
        print("skipping")
    if successes == 50:
        break
        

FC(F)(F)c1nnc(S[C@H]2CC[C@H]3C[C@H]32)[nH]1
COC(=O)[C@]12CC[C@H](C1=O)[C@@H](O)C[C@@H]2c1ccco1
NS(=O)(=O)N[C@@H]1OC[C@@H](O)[C@H](O)[C@@H]1O
FC(F)(F)c1ccc(CN2C[C@H]3CC4CC3(C4)C2)cc1
N=c1c2cn[nH]c2ncn1-c1ccc(-c2ccc(F)cc2)s1
skipping
O=c1ncc(F)cn1[C@H]1CC[C@H](N[SH](=O)=O)C1
O=C1O[C@H](CO)[C@@H](c2ccccc2)N1c1cccc(F)c1
O=CN[C@@H]1C[C@@H](C=O)N(S(=O)(=O)c2ccccc2)C1
O=c1c2ncn([C@@H]3CO[C@@H](CO)O3)c2nc2[nH]ccn12
O=C(c1ccccc1F)N1CCN(C(=O)[C@H]2CCCN2)CC1
O=C(c1ccc(Oc2ccccn2)cc1)c1nc2ncccc2[nH]1
CC1(C)[C@@H](c2cccc(Cl)c2)[C@@H]1c1c[nH]cn1
C[C@H]1N=C(c2ccccc2Cl)c2c(N=N)cccc2NC1=O
skipping
c1ccc2c(c1)O[C@H](c1ccc(-c3cn[nH]c3)cc1)n1c-2cc2ccccc21
O=C1c2ccccc2C(=O)C1C[C@]1(c2cccc(Cl)c2)CO1
O=C1C[C@@H](CC(=O)[C@@H]2CCCCC2=O)C(=O)N1
COc1ccn(-c2ccc3c4c([nH]c3n2)C[C@@H]2CC[C@H]4N2)c(=O)c1
COc1cccc([C@H]2OCC(=O)Nc3ccc(Cl)cc32)c1OC
OC[C@@H]1O[C@@H](n2cc(F)c3cncnc32)[C@@H](O)[C@@H]1O
O=C1C=C(c2cc3ccccc3s2)C[C@H](c2ccccc2)C1
O=c1[nH]cnc2nc(-c3ccc(P(=O)(O)O)cc3)[nH]c12
CN(C(=O)Cn1ncccc1=O)[C@@H]1CCS(=O)(=O)

In [13]:
!cat gen_conf_{0..49}.sdf > inputs.sdf


In [14]:
input_file_name = "inputs.sdf"
#import urllib
#urllib.request.urlretrieve("https://raw.githubusercontent.com/openforcefield/openff-interchange/refs/heads/main/openff/interchange/_tests/data/MiniDrugBankTrimmed.sdf",
#                           input_file_name)
#orig_input_path = get_data_file_path(f"data/{input_file_name}")
#shutil.copyfile(orig_input_path, input_file_name)
#subprocess.run(["tar", "xvzf", input_file_name])

In [1]:
from openff.toolkit import Molecule, ForceField
from tqdm import tqdm
mols = Molecule.from_file("inputs.sdf")
ff = ForceField("openff_unconstrained-2.3.0-rc2.offxml")
for idx, mol in enumerate(tqdm(mols, desc="Processing molecules")):
    interchange = ff.create_interchange(mol.to_topology())
    interchange.minimize()
    mol.conformers[0] = interchange.positions
    mol.to_file(f"input_min_{idx}.sdf", file_format="sdf")

LICENSE: Could not open license file specified by OE_LICENSE environment variable "/Users/jeffreywagner/oe_license.txt"
LICENSE: Could not open license file "oe_license.txt" in local directory
LICENSE: N.B. OE_DIR environment variable is not set
LICENSE: No product keys!
LICENSE: No product keys!
LICENSE: No product keys!
The OpenEye Toolkits are found to be installed but not licensed and therefore will not be used.
The OpenEye Toolkits require a (free for academics) license, see https://docs.eyesopen.com/toolkits/python/quickstart-python/license.html
LICENSE: No product keys!
The OpenEye Toolkits are found to be installed but not licensed and therefore will not be used.
The OpenEye Toolkits require a (free for academics) license, see https://docs.eyesopen.com/toolkits/python/quickstart-python/license.html
/Users/jeffreywagner/micromamba/envs/off-tk-dev/lib/python3.12/site-packages/openff/amber_ff_ports/amber_ff_ports.py:8: UserWarning: pkg_resources is deprecated as an API. See https:

In [2]:
! cat input_min_{0..49}.sdf > minimized_inputs.sdf